# GoalT (Goal Tree) -- Interactive Demo

A multi-parent, value-propagating goal graph. This notebook is meant to be run top to bottom -- click `Runtime > Run all` in Colab (or run cells one by one) and watch the tree re-balance itself as you add goals.

**What you're looking at:**
- One root goal holds a value of `1.0`.
- Every goal splits its value across its children (weights always sum to 1.0 per parent).
- A goal can have **multiple parents** -- so it accumulates value from each of them.
- Add a new goal and only the affected part of the graph recomputes (not the whole tree).
- Cycles are rejected outright, not silently allowed to loop.

This is a concept-stage project, not a finished product. Known open questions are listed at the bottom. If you find a case that breaks it, that's useful -- please open an issue.

In [ ]:
# If running in Colab, this pulls the two small source files from the repo.
# If you already cloned the repo locally, you can skip this cell.
import os
if not os.path.exists('goal_tree.py'):
    !pip install networkx matplotlib -q
    !wget -q https://raw.githubusercontent.com/furkanYanteri1/GOALT-the-goal-tree/main/goal_tree.py
    !wget -q https://raw.githubusercontent.com/furkanYanteri1/GOALT-the-goal-tree/main/visualize.py
print('ready')

In [ ]:
from goal_tree import GoalGraph, CycleError
from visualize import draw

g = GoalGraph()
g.add_root('root', 'Ship v2 of the product')
g.add_goal('a', 'Improve onboarding', parents=['root'])
g.add_goal('b', 'Improve performance', parents=['root'])
g.add_goal('c', 'Fix export bug', parents=['a', 'b'])  # multi-parent: depends on both

print(g)
draw(g, title='Initial goal tree')

Notice `c` (Fix export bug) already ranks above `a` and `b` individually -- because it's a child of *both*, it accumulates value from each parent. That's the core idea: **the goal with the most real dependencies pulls the most value, automatically.**

Now let's add a new goal under `a` and see the tree re-balance:

In [ ]:
g.add_goal('d', 'Add onboarding checklist', parents=['a'])
print(g)
draw(g, title='After adding a new goal under "Improve onboarding"')

`a`'s value (0.5) is now split between `c` and `d`, which is why `c`'s total value dropped from 1.0 to 0.75 -- it's still getting the full 0.5 from `b`, but only half of `a`'s 0.5 now.

## Cycles are rejected, not silently looped

Multi-parent graphs make cycles easy to introduce by accident. This implementation checks for that explicitly on every edge and refuses to add one that would break the DAG property.

In [ ]:
try:
    g.add_goal('e', 'This would loop back', parents=['c'])
    g.g.add_edge('e', 'root')  # manually forcing an illegal edge to prove the point
except CycleError as err:
    print('Rejected as expected:', err)

## Plugging in an LLM for redistribution (optional)

By default, `GoalGraph` splits value equally among siblings -- deterministic, no API key needed, always converges. You can replace that with any policy, including an LLM that decides weights based on context ("the user said speed matters more than polish this sprint", etc).

The engine never trusts the LLM's numbers directly -- whatever it returns gets validated and re-normalized so the graph stays mathematically consistent even if the model returns something odd.

Below is a mock LLM (a Python function that fakes a response) just so this cell runs without any API key. Swap `mock_llm_call` for a real call to your model of choice -- see `goal_tree.make_llm_redistribution_fn` in the source.

In [ ]:
import json
from goal_tree import make_llm_redistribution_fn

def mock_llm_call(prompt: str) -> str:
    # Pretend the model always favors whichever child sounds more urgent.
    # Replace this with a real call, e.g.:
    #   response = client.messages.create(model=..., temperature=0, messages=[...])
    #   return response.content[0].text
    if 'bug' in prompt.lower():
        return json.dumps({'c': 3, 'd': 1})
    return json.dumps({k: 1 for k in ['c', 'd']})

llm_fn = make_llm_redistribution_fn(mock_llm_call)

g2 = GoalGraph(redistribution_fn=llm_fn)
g2.add_root('root', 'Ship v2 of the product')
g2.add_goal('a', 'Improve onboarding', parents=['root'])
g2.add_goal('b', 'Improve performance', parents=['root'])
g2.add_goal('c', 'Fix export bug', parents=['a', 'b'])
g2.add_goal('d', 'Add onboarding checklist', parents=['a'])

print(g2)
draw(g2, title='LLM-weighted redistribution (mocked model)')

## Try it yourself

Edit the cell below -- build your own goal tree, add a few goals with multiple parents, and see what floats to the top.

In [ ]:
my_graph = GoalGraph()
my_graph.add_root('root', 'YOUR MAIN GOAL HERE')
# my_graph.add_goal('id', 'label', parents=['root'])
# my_graph.add_goal('id2', 'label2', parents=['id', 'root'])  # multi-parent example

print(my_graph)
draw(my_graph, title='Your goal tree')

## Known open questions (honest list, not marketing)

- **Convergence with LLM-driven weights**: the deterministic fallback always converges. Whether repeated LLM-driven re-weighting stays stable over many edits, on a large graph, is not yet proven -- just observed on small examples.
- **Global value is not conserved**: because a node can have multiple parents, the sum of all node values in the graph is *not* 1.0 overall -- only locally, per parent, do children's weights sum to 1.0. This is intentional (it's how "more parents = more pull" works) but worth understanding before reading too much into raw values.
- **Cost at scale**: if you plug in a real LLM, every `add_goal` call triggers one redistribution call per affected parent. On a large, deep graph this could mean many API calls per edit. Caching / batching is on the roadmap, not yet implemented.
- **No benchmark yet**: this hasn't been tested against classical prioritization methods (AHP, weighted scoring, plain OKRs) on real backlogs. That comparison is the next step, not a claim already made.

If you want to help with any of these, the repo is open -- issues and PRs welcome.